In [3]:
import time
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

print("⏳ Khởi động SparkSession cho User Features...")
start_time = time.time()

# Bổ sung cấu hình Iceberg Catalog giống như lúc bạn tạo bảng Silver
spark = SparkSession.builder \
    .appName("Amazon_User_Features_Gold") \
    .master("local[4]") \
    .config("spark.driver.memory", "16g") \
    .config("spark.sql.shuffle.partitions", "32") \
    .config("spark.sql.catalog.gcp_catalog", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.gcp_catalog.type", "hadoop") \
    .config("spark.sql.catalog.gcp_catalog.warehouse", "gs://amazon-reviews-lakehouse-warehouse/warehouse/") \
    .getOrCreate()

print("✅ SparkSession khởi tạo thành công (Đã bật Iceberg)!")

⏳ Khởi động SparkSession cho User Features...
✅ SparkSession khởi tạo thành công (Đã bật Iceberg)!


26/06/14 19:40:36 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [4]:
print("⏳ Đang đọc dữ liệu Silver từ Iceberg Catalog...")

# Đọc trực tiếp bảng Iceberg qua catalog, không đọc file vật lý
df_silver = spark.read.table("gcp_catalog.silver.amazon_reviews_cleaned")

# Lệnh action ép Spark quét dữ liệu kiểm tra
total_rows = df_silver.count()
print(f"✅ Đọc thành công! Tổng số dòng trong Silver: {total_rows}")
df_silver.printSchema()

⏳ Đang đọc dữ liệu Silver từ Iceberg Catalog...
✅ Đọc thành công! Tổng số dòng trong Silver: 430143811
root
 |-- asin: string (nullable = true)
 |-- helpful_vote: long (nullable = true)
 |-- parent_asin: string (nullable = true)
 |-- rating: double (nullable = true)
 |-- text: string (nullable = true)
 |-- timestamp: long (nullable = true)
 |-- title: string (nullable = true)
 |-- verified_purchase: boolean (nullable = true)
 |-- ingest_ts: timestamp (nullable = true)
 |-- user_id_hashed: string (nullable = true)



In [5]:
print("🚀 Bắt đầu tính toán đặc trưng thống kê người dùng...")

# 1. Tính toán thống kê và lấy Timestamp mới nhất
df_user_features = df_silver.groupBy("user_id_hashed").agg(
    F.count("rating").alias("total_reviews"),
    F.round(F.avg("rating"), 2).alias("avg_rating_given"),
    F.round(F.stddev("rating"), 2).alias("stddev_rating_given"),
    # Chuyển đổi unix timestamp (millisecond) sang kiểu Timestamp chuẩn cho Feast
    F.from_unixtime(F.max("timestamp") / 1000).cast("timestamp").alias("event_timestamp")
)

# 2. Xử lý Null cho độ lệch chuẩn (Trường hợp user chỉ có 1 review)
df_user_features = df_user_features.fillna({"stddev_rating_given": 0.0})

# 3. Đổi tên cột chuẩn hóa entity (Feast Entity)
df_user_features = df_user_features.withColumnRenamed("user_id_hashed", "user_id")

print("📊 Preview 5 dòng dữ liệu đầu tiên:")
df_user_features.show(5, truncate=False)

🚀 Bắt đầu tính toán đặc trưng thống kê người dùng...
📊 Preview 5 dòng dữ liệu đầu tiên:


+----------------------------------------------------------------+-------------+----------------+-------------------+-------------------+
|user_id                                                         |total_reviews|avg_rating_given|stddev_rating_given|event_timestamp    |
+----------------------------------------------------------------+-------------+----------------+-------------------+-------------------+
|f3f140ef3411c590221bfc2efda8cbf08022e029cb0a9acde18994d53a862479|33           |3.94            |1.48               |2003-12-12 11:15:03|
|4e8d79a0c8a2784950ecc4fdb13af9ec7d4225e5b10441b77453df7946b2c695|9            |2.67            |1.66               |2014-11-04 21:47:51|
|af2d9039e3bca02f4a39ae9b120b77b730b608626cead645eae1c0f195a42753|17           |4.12            |1.17               |2010-12-25 00:44:51|
|0c71b49b9f36aabbdc0a70778eca7c51e65bde9f99189fee3ea5f46a33dc93b2|49           |2.08            |1.32               |2019-08-21 22:10:56|
|c54938c9553514556e28bff1b01e37f9b

In [6]:
gold_user_path = "gs://amazon-reviews-lakehouse-warehouse/warehouse/gold/user_features"
print(f"⏳ Đang phân mảnh lại (repartition=16) và ghi xuống GCS tại: {gold_user_path}")

# Repartition để tối ưu hóa kích thước file Parquet, tránh việc sinh ra quá nhiều file rác nhỏ
df_user_features.repartition(16).write \
    .mode("overwrite") \
    .parquet(gold_user_path)
    
end_time = time.time()
print(f"✅ HOÀN THÀNH! Dữ liệu User Features đã sẵn sàng cho Feast.")
print(f"⏱️ Tổng thời gian chạy luồng: {round(end_time - start_time, 2)} giây.")

⏳ Đang phân mảnh lại (repartition=16) và ghi xuống GCS tại: gs://amazon-reviews-lakehouse-warehouse/warehouse/gold/user_features


✅ HOÀN THÀNH! Dữ liệu User Features đã sẵn sàng cho Feast.
⏱️ Tổng thời gian chạy luồng: 1056.21 giây.
